# Klasifikasi Jenis Kerusakan Mesin Tanpa Function

Notebook ini menampilkan kode lengkap secara berurutan tanpa membuat function khusus. Alurnya dimulai dari import library, load data, cleaning, preprocessing, modelling, hingga evaluasi.

## 1. Import Library

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

## 2. Load Dataset

In [ ]:
RANDOM_STATE = 42
CSV_PATH = Path(r"/content/ai4i2020.csv")
OUTPUT_DIR = Path("artifacts/no_function_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print("Ukuran dataset:", df.shape)
df.head()

Ukuran dataset: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 3. Cleaning dan Pembentukan Target

Dataset AI4I 2020 memiliki indikator kerusakan `TWF`, `HDF`, `PWF`, `OSF`, dan `RNF`. Pada bagian ini indikator tersebut diubah menjadi target multikelas bernama `Failure Type`.

In [ ]:
print("Jumlah missing value per kolom:")
display(df.isna().sum())

failure_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]
failure_names = {
    "TWF": "Tool Wear Failure",
    "HDF": "Heat Dissipation Failure",
    "PWF": "Power Failure",
    "OSF": "Overstrain Failure",
    "RNF": "Random Failures",
}

failure_count = df[failure_columns].sum(axis=1)
df["Failure Type"] = "No Failure"

for column_name, failure_name in failure_names.items():
    df.loc[(df[column_name] == 1) & (failure_count == 1), "Failure Type"] = failure_name

df.loc[failure_count > 1, "Failure Type"] = "Multiple Failures"

print("Distribusi target:")
display(df["Failure Type"].value_counts())

Jumlah missing value per kolom:


,0
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


Distribusi target:


,count
Failure Type,
No Failure,9652
Heat Dissipation Failure,106
Power Failure,80
Overstrain Failure,78
Tool Wear Failure,42
Multiple Failures,24
Random Failures,18


## 4. Exploratory Data Analysis

In [ ]:
numeric_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

display(df[numeric_columns].describe().T)

class_distribution = df["Failure Type"].value_counts().rename_axis("Failure Type").reset_index(name="Count")
class_distribution["Percentage"] = class_distribution["Count"] / len(df) * 100
display(class_distribution)

,count,mean,std,min,25%,50%,75%,max
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.0,108.0,162.0,253.0


,Failure Type,Count,Percentage
0,No Failure,9652,96.52
1,Heat Dissipation Failure,106,1.06
2,Power Failure,80,0.80
3,Overstrain Failure,78,0.78
4,Tool Wear Failure,42,0.42
5,Multiple Failures,24,0.24
6,Random Failures,18,0.18


## 5. Pemisahan Fitur dan Target

In [ ]:
drop_columns = ["UDI", "Product ID", "Target", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF", "Failure Type"]

X_raw = df.drop(columns=drop_columns, errors="ignore")
y = df["Failure Type"]

print("Fitur awal:")
display(X_raw.head())
print("Jumlah fitur awal:", X_raw.shape[1])

Fitur awal:


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


Jumlah fitur awal: 6


## 6. Feature Engineering

In [ ]:
X_engineered = X_raw.copy()

air_temp = X_engineered["Air temperature [K]"]
process_temp = X_engineered["Process temperature [K]"]
rotational_speed = X_engineered["Rotational speed [rpm]"].replace(0, np.nan)
torque = X_engineered["Torque [Nm]"]
tool_wear = X_engineered["Tool wear [min]"]

X_engineered["Temperature difference [K]"] = process_temp - air_temp
X_engineered["Temperature ratio"] = process_temp / air_temp
X_engineered["Torque per rpm"] = torque / rotational_speed
X_engineered["Estimated power [W]"] = torque * rotational_speed / 9.5488
X_engineered["Tool wear torque"] = tool_wear * torque
X_engineered["Tool wear speed"] = tool_wear * rotational_speed
X_engineered["Heat dissipation risk"] = ((X_engineered["Temperature difference [K]"] <= 8.6) & (rotational_speed < 1380)).astype(int)
X_engineered["Low power risk"] = (X_engineered["Estimated power [W]"] < 3500).astype(int)
X_engineered["High power risk"] = (X_engineered["Estimated power [W]"] > 9000).astype(int)
X_engineered["Tool wear risk"] = (tool_wear >= 198).astype(int)
X_engineered["Overstrain risk"] = (
    ((X_engineered["Type"] == "L") & (X_engineered["Tool wear torque"] > 11000))
    | ((X_engineered["Type"] == "M") & (X_engineered["Tool wear torque"] > 12000))
    | ((X_engineered["Type"] == "H") & (X_engineered["Tool wear torque"] > 13000))
).astype(int)

print("Jumlah fitur setelah feature engineering:", X_engineered.shape[1])
display(X_engineered.head())

Jumlah fitur setelah feature engineering: 17


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temperature difference [K],Temperature ratio,Torque per rpm,Estimated power [W],Tool wear torque,Tool wear speed,Heat dissipation risk,Low power risk,High power risk,Tool wear risk,Overstrain risk
0,M,298.1,308.6,1551,42.8,0,10.5,1.035223,0.027595,6951.952078,0.0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,10.5,1.035211,0.032884,6827.077748,138.9,4224,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,10.4,1.034888,0.032977,7749.790550,247.0,7490,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,10.4,1.034876,0.027565,5927.812919,276.5,10031,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,10.5,1.035211,0.028409,5898.123324,360.0,12672,0,0,0,0,0


## 7. Encoding Target dan Split Data

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_.tolist()

train_index, test_index = train_test_split(
    np.arange(len(y_encoded)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

print("Nama kelas:", class_names)
print("Jumlah data train:", len(train_index))
print("Jumlah data test:", len(test_index))

Nama kelas: ['Heat Dissipation Failure', 'Multiple Failures', 'No Failure', 'Overstrain Failure', 'Power Failure', 'Random Failures', 'Tool Wear Failure']
Jumlah data train: 8000
Jumlah data test: 2000


## 8. Preprocessing

Preprocessing terdiri dari imputasi median dan standardisasi untuk fitur numerik, serta imputasi modus dan one-hot encoding untuk fitur kategorikal.

In [ ]:
feature_sets = {
    "raw": X_raw,
    "engineered": X_engineered,
}

samplers = {
    "none": None,
    "smote": SMOTE(k_neighbors=2, random_state=RANDOM_STATE),
    "smote_tomek": SMOTETomek(smote=SMOTE(k_neighbors=2, random_state=RANDOM_STATE), random_state=RANDOM_STATE),
}

models = {
    "logistic_regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=16,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "xgboost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

## 9. Modelling dan Evaluasi Ablation Study

In [ ]:
results = []
per_class_results = []
best_result = None

for feature_set_name, X_data in feature_sets.items():
    X_train = X_data.iloc[train_index].reset_index(drop=True)
    X_test = X_data.iloc[test_index].reset_index(drop=True)
    y_train = y_encoded[train_index]
    y_test = y_encoded[test_index]

    categorical_features = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

    numeric_pipeline = ImbPipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = ImbPipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numerical_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    for sampler_name, sampler in samplers.items():
        for model_name, model in models.items():
            steps = [("preprocessor", preprocessor)]
            if sampler is not None:
                steps.append(("sampler", sampler))
            steps.append(("model", model))

            pipeline = ImbPipeline(steps=steps)

            print(f"Training: {feature_set_name} | {sampler_name} | {model_name}")
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
            f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
            f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)
            precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
            recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)

            report = classification_report(
                y_test,
                y_pred,
                labels=list(range(len(class_names))),
                target_names=class_names,
                output_dict=True,
                zero_division=0,
            )

            result_row = {
                "feature_set": feature_set_name,
                "sampler": sampler_name,
                "model": model_name,
                "accuracy": accuracy,
                "balanced_accuracy": balanced_accuracy,
                "f1_macro": f1_macro,
                "f1_weighted": f1_weighted,
                "precision_macro": precision_macro,
                "recall_macro": recall_macro,
            }
            results.append(result_row)

            for class_name in class_names:
                per_class_results.append(
                    {
                        "feature_set": feature_set_name,
                        "sampler": sampler_name,
                        "model": model_name,
                        "class": class_name,
                        "precision": report[class_name]["precision"],
                        "recall": report[class_name]["recall"],
                        "f1_score": report[class_name]["f1-score"],
                        "support": report[class_name]["support"],
                    }
                )

            if best_result is None or f1_macro > best_result["metrics"]["f1_macro"]:
                best_result = {
                    "metrics": result_row,
                    "classification_report": report,
                    "confusion_matrix": confusion_matrix(
                        y_test,
                        y_pred,
                        labels=list(range(len(class_names))),
                    ).tolist(),
                }

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
per_class_df = pd.DataFrame(per_class_results)

display(results_df)

Training: raw | none | logistic_regression
Training: raw | none | random_forest
Training: raw | none | xgboost
Training: raw | smote | logistic_regression
Training: raw | smote | random_forest
Training: raw | smote | xgboost
Training: raw | smote_tomek | logistic_regression
Training: raw | smote_tomek | random_forest
Training: raw | smote_tomek | xgboost
Training: engineered | none | logistic_regression
Training: engineered | none | random_forest
Training: engineered | none | xgboost
Training: engineered | smote | logistic_regression
Training: engineered | smote | random_forest
Training: engineered | smote | xgboost
Training: engineered | smote_tomek | logistic_regression
Training: engineered | smote_tomek | random_forest
Training: engineered | smote_tomek | xgboost


,feature_set,sampler,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
13,engineered,smote,random_forest,0.9600,0.780385,0.728591,0.974148,0.723525,0.780385
16,engineered,smote_tomek,random_forest,0.9600,0.780385,0.728591,0.974148,0.723525,0.780385
10,engineered,none,random_forest,0.9895,0.713620,0.713509,0.988754,0.713399,0.713620
14,engineered,smote,xgboost,0.9785,0.711991,0.712686,0.983197,0.713389,0.711991
17,engineered,smote_tomek,xgboost,0.9785,0.711991,0.712686,0.983197,0.713389,0.711991
12,engineered,smote,logistic_regression,0.6340,0.838897,0.694477,0.768980,0.721023,0.838897
15,engineered,smote_tomek,logistic_regression,0.6340,0.838897,0.694477,0.768980,0.721023,0.838897
11,engineered,none,xgboost,0.9915,0.648066,0.653899,0.988932,0.662266,0.648066
9,engineered,none,logistic_regression,0.6280,0.873649,0.613981,0.762864,0.618464,0.873649
2,raw,none,xgboost,0.9845,0.545027,0.586779,0.980901,0.647489,0.545027


## 10. Hasil Model Terbaik

In [ ]:
print("Konfigurasi terbaik berdasarkan macro F1-score:")
display(pd.DataFrame([best_result["metrics"]]))

Konfigurasi terbaik berdasarkan macro F1-score:


,feature_set,sampler,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,engineered,smote,random_forest,0.96,0.780385,0.728591,0.974148,0.723525,0.780385


In [ ]:
best_report_df = pd.DataFrame(best_result["classification_report"]).T
display(best_report_df)

,precision,recall,f1-score,support
Heat Dissipation Failure,1.000000,1.000000,1.000000,21.00
Multiple Failures,1.000000,1.000000,1.000000,5.00
No Failure,0.995713,0.962694,0.978925,1930.00
Overstrain Failure,1.000000,1.000000,1.000000,16.00
Power Failure,1.000000,1.000000,1.000000,16.00
Random Failures,0.000000,0.000000,0.000000,4.00
Tool Wear Failure,0.068966,0.500000,0.121212,8.00
accuracy,0.960000,0.960000,0.960000,0.96
macro avg,0.723525,0.780385,0.728591,2000.00
weighted avg,0.990139,0.960000,0.974148,2000.00


In [ ]:
confusion_df = pd.DataFrame(
    best_result["confusion_matrix"],
    index=class_names,
    columns=class_names,
)
display(confusion_df)

,Heat Dissipation Failure,Multiple Failures,No Failure,Overstrain Failure,Power Failure,Random Failures,Tool Wear Failure
Heat Dissipation Failure,21,0,0,0,0,0,0
Multiple Failures,0,5,0,0,0,0,0
No Failure,0,0,1858,0,0,18,54
Overstrain Failure,0,0,0,16,0,0,0
Power Failure,0,0,0,0,16,0,0
Random Failures,0,0,4,0,0,0,0
Tool Wear Failure,0,0,4,0,0,0,4


## 11. Simpan Hasil Eksperimen

In [ ]:
results_df.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)
per_class_df.to_csv(OUTPUT_DIR / "per_class_metrics.csv", index=False)
confusion_df.to_csv(OUTPUT_DIR / "best_confusion_matrix.csv")

with (OUTPUT_DIR / "ablation_summary.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "class_names": class_names,
            "train_size": int(len(train_index)),
            "test_size": int(len(test_index)),
            "best_result": best_result,
            "raw_features": X_raw.columns.tolist(),
            "engineered_features": X_engineered.columns.tolist(),
        },
        file,
        indent=2,
    )

print("Hasil eksperimen tersimpan di:", OUTPUT_DIR)

Hasil eksperimen tersimpan di: artifacts/no_function_notebook
